# Robustness of the CO-fraction-vs-shear trend

Checks from `EK_ideas.md` ("CO fraction vs shear") applied to the three shear
results of `reproduce_results_check_co_mom0.ipynb`:

* **Result 2**: `ana.result_co_vs_shear(...)`, the raw CO-bright fraction vs |shear|
* **Result 3**: `ana.resolution_study(...)`, the same curve at each smoothing FWHM
* **Result 4**: `ana.shear_sensitivity(...)`, normalized, with the wedge block-bootstrap band and `rel_drop`
* **Result 3 (divergence)**: the CO fraction vs divergence curves from the same `resolution_study` calls (last section)

Each check is a pixel selection (`sel=`, built with `ana.selection_mask(...)`) or a
displaced CO map (`co_mode=`), and is passed unchanged to all three calls:

* **A. HI Laplacian sign** (Josh). Is the lack of CO at high shear beam smearing
  between offset components at HI column jumps? Split on the sign of ∇²Σ_HI.
  ∇² < 0 marks HI peaks/ridges, and ∇² > 0 marks troughs and the flanks of jumps.
* **B. HI velocity dispersion.** Large HI mom2 flags blended or offset components.
  The trend should survive in the low-dispersion pixels.
* **C. Displaced CO.** Rotate or mirror the CO map (as in the divergence null test). A local
  CO–shear link should vanish, and whatever survives is set by shared large-scale structure.

Setup is identical to the mom0 notebook (same HI, CO map and config), plus the HI mom2 map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from exgal_flows import FlowAnalysis, plotting

DATA = "/arc/projects/VVINGS/exgal_flows"
HI_NEW = f"{DATA}/HI/lglbs_ABCD"
MODEL = f"{DATA}/rot_model/rad.mod.fits"
# same CO map as reproduce_results_check_co_mom0.ipynb
CO_PEAK = f"{DATA}/CO/m33_all_co21_2p5kms_8p0as_aligned_strict_mom0.fits"

ana = FlowAnalysis.from_files(
    mom1_path=f"{HI_NEW}/m33_A+B+C+D+GBT_hi21cm_strict_mom1.fits",
    mom0_path=f"{HI_NEW}/m33_A+B+C+D+GBT_hi21cm_strict_mom0.fits",
    model_path=MODEL,
    co_paths={"hires": CO_PEAK},
    extra_paths={"hi_mom2": f"{HI_NEW}/m33_A+B+C+D+GBT_hi21cm_strict_mom2.fits"},
    # NOTE: the CO map is mom0 (K km/s) but m33.yaml's co_threshold_K is a peak-Tb
    # threshold -- same choice as the mom0 notebook; see config/m33_co_mom0.yaml.
    # config_path="../config/m33_co_mom0.yaml",
    config_path="../config/m33.yaml",
)
print(f"pixel scale: {ana.pixel_scale_pc:.2f} pc/pix")

DECIM = 2
N_BOOT = 250

In [ ]:
import warnings


def compare(curves, title, do_r3=True, colors=None):
    # curves: {label: dict(sel=..., co_mode=..., **extra shear_sensitivity kwargs)}
    labels = list(curves)
    colors = colors or [f"C{i}" for i in range(len(labels))]
    an = ana.config.analysis
    sl = (slice(None, None, DECIM), slice(None, None, DECIM))
    base = (ana.radius[sl] < ana._radius_max_pix) & np.isfinite(np.asarray(ana.co_deproj)[sl])

    r2, r3, r4 = {}, {}, {}
    for lab, kw in curves.items():
        sel, co_mode = kw.get("sel"), kw.get("co_mode")
        r2[lab] = ana.result_co_vs_shear(decim=DECIM, sel=sel, co_mode=co_mode)
        r4[lab] = ana.shear_sensitivity(decim=DECIM, n_boot=N_BOOT, **kw)
        if do_r3:
            r3[lab] = ana.resolution_study(decim=DECIM, sel=sel, co_mode=co_mode)

    # Result 2 (raw) and Result 4 (normalized, with bands)
    fig, (a2, a4) = plt.subplots(1, 2, figsize=(11, 4))
    for (lab, cur), c in zip(r2.items(), colors):
        plotting.plot_co_fraction_vs_gradient(cur, ax=a2, xlabel="|shear| [km/s/kpc]",
                                              color=c, marker="o", ms=3, label=lab)
    a2.set_title("Result 2: CO-bright fraction")
    a2.legend(frameon=False, fontsize=8)
    plotting.plot_shear_sensitivity(r4, ax=a4, colors=colors)
    a4.set_title("Result 4: normalized, block-bootstrap band")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

    # Result 3 (per smoothing scale)
    if do_r3:
        n = len(an.smoothing_fwhm_pc)
        fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2), sharey=True)
        for (lab, st), c in zip(r3.items(), colors):
            plotting.plot_resolution_study(st, which="shear", axes=axes, label=lab, color=c)
        fig.suptitle(f"Result 3: {title}", y=1.02)
        plt.show()

    # table
    print(f"{'selection':>22s} {'pix kept':>8s} {'lo_frac':>8s} {'hi_frac':>8s}   rel_drop [68%]")
    for lab, kw in curves.items():
        sel = kw.get("sel")
        kept = 1.0 if sel is None else np.count_nonzero(sel[sl] & base) / np.count_nonzero(base)
        s = r4[lab]
        print(f"{lab:>22s} {kept:8.2f} {s.lo_frac:8.3f} {s.hi_frac:8.3f}   "
              f"{s.rel_drop:5.2f} [{s.rel_drop_lo:.2f}, {s.rel_drop_hi:.2f}]")
    return {"r2": r2, "r3": r3, "r4": r4, "colors": colors, "title": title}


# ---- spatial view of a selection -------------------------------------------
# Footprint = HI mom0 kept after matching to the CO (both finite) inside the
# analysis aperture.  Maps are clipped to its bounding box and drawn as the
# fraction of footprint pixels selected in MAP_BLOCK_PC blocks, so a
# pixel-level (speckled) mask reads as where the selection dominates.
MAP_BLOCK_PC = 50.0
FOOT = (np.isfinite(np.asarray(ana.co_deproj)) & np.isfinite(np.asarray(ana.mom0_deproj))
        & (ana.radius < ana._radius_max_pix))
_r, _c = np.flatnonzero(FOOT.any(axis=1)), np.flatnonzero(FOOT.any(axis=0))
BBOX = (slice(_r[0], _r[-1] + 1), slice(_c[0], _c[-1] + 1))
_B = max(1, int(round(MAP_BLOCK_PC / ana.pixel_scale_pc)))
_cy, _cx = np.unravel_index(np.argmin(ana.radius), ana.radius.shape)


def _blocks(a):
    # nan-mean over _B x _B blocks of the clipped, footprint-masked map
    a = np.where(FOOT, np.asarray(a, dtype=float), np.nan)[BBOX]
    ny, nx = (a.shape[0] // _B) * _B, (a.shape[1] // _B) * _B
    a = a[:ny, :nx].reshape(ny // _B, _B, nx // _B, _B)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a, axis=(1, 3))


_kpc = ana.pixel_scale_pc / 1000.0
_M0 = _blocks(ana.mom0_deproj)
_ny, _nx = _M0.shape
EXTENT = [(_c[0] - _cx) * _kpc, (_c[0] + _nx * _B - _cx) * _kpc,
          (_r[0] - _cy) * _kpc, (_r[0] + _ny * _B - _cy) * _kpc]
M0_LEVELS = np.nanpercentile(_M0, [50, 80, 95])
CO_BRIGHT = _blocks(np.asarray(ana.co_deproj) > ana.config.analysis.co_threshold_K)


def _hi_contours(ax):
    ax.contour(_M0, levels=M0_LEVELS, extent=EXTENT, origin="lower",
               colors="darkorange", linewidths=[0.6, 0.9, 1.3])
    ax.set_xlabel("x [kpc]")
    ax.set_ylabel("y [kpc]")


def show_masks(masks, ncols=2):
    # first panel: CO-bright fraction (reference); then one panel per mask
    panels = {"CO-bright fraction (reference)": (CO_BRIGHT, "Greys")}
    panels.update({lab: (_blocks(m), "Blues") for lab, m in masks.items()})
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 6.0 * nrows), squeeze=False)
    for ax, (lab, (img, cmap)) in zip(axes.ravel(), panels.items()):
        im = ax.imshow(img, origin="lower", extent=EXTENT, cmap=cmap, vmin=0, vmax=1,
                       interpolation="nearest")
        ax.set_facecolor("0.8")          # grey = outside the HI+CO footprint
        _hi_contours(ax)
        ax.set_title(lab)
        fig.colorbar(im, ax=ax, shrink=0.8,
                     label=f"fraction of pixels per {MAP_BLOCK_PC:.0f} pc block")
    for ax in axes.ravel()[len(panels):]:
        ax.set_visible(False)
    fig.suptitle(f"HI mom0 contours: {', '.join(f'{v:.0f}' for v in M0_LEVELS)} "
                 "(50/80/95th pct, deprojected units); grey = outside HI+CO footprint", y=1.0)
    fig.tight_layout()
    plt.show()

## A. HI Laplacian sign

∇²Σ_HI is computed from the deprojected HI moment-0 map after a NaN-aware Gaussian
smoothing (`laplacian_fwhm_pc`). The native-resolution Laplacian is dominated by
pixel noise. The default here, 50 pc, is the bottom of the 50–100 pc range the
shear result is claimed on. The second cell sweeps the scale.

If the CO deficit at high shear comes from beam smearing at HI jumps, it should
be concentrated in ∇² > 0 and weaken or vanish for ∇² < 0.

In [ ]:
LAP_FWHM_PC = 50.0
lap_pos = ana.selection_mask(laplacian_sign=+1, laplacian_fwhm_pc=LAP_FWHM_PC)
lap_neg = ana.selection_mask(laplacian_sign=-1, laplacian_fwhm_pc=LAP_FWHM_PC)
show_masks({"∇²Σ_HI > 0": lap_pos, "∇²Σ_HI < 0": lap_neg})

lap = compare({"all": {},
               "∇²Σ_HI > 0": {"sel": lap_pos},
               "∇²Σ_HI < 0": {"sel": lap_neg}},
              f"HI Laplacian sign (smoothed to {LAP_FWHM_PC:.0f} pc)",
              colors=["k", "C3", "C0"])

In [ ]:
# sensitivity to the Laplacian smoothing scale (Result 4 only)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, sign, name in zip(axes, (+1, -1), ("∇²Σ_HI > 0", "∇²Σ_HI < 0")):
    sweep = {f"{fw:g} pc" if fw else "native":
             ana.shear_sensitivity(decim=DECIM, n_boot=N_BOOT,
                                   sel=ana.selection_mask(laplacian_sign=sign, laplacian_fwhm_pc=fw))
             for fw in (0.0, 25.0, 50.0, 100.0)}
    plotting.plot_shear_sensitivity(sweep, ax=ax, band=False)
    ax.set_title(name)
    print(name, {k: round(v.rel_drop, 2) for k, v in sweep.items()})
fig.suptitle("Laplacian smoothing scale")
fig.tight_layout()
plt.show()

## B. HI velocity dispersion

The HI strict mom2 map (km/s, observed, so no cos i factor) is split into tertiles over the pixels
used by the shear results (in the aperture, with CO coverage). Beam smearing between offset components
inflates mom2, so the trend should persist in the lowest tertile. Change `DISP_EDGES`
for fixed cuts (e.g. `[0, 8, 12, np.inf]`).

In [ ]:
disp = ana.extra_deproj("hi_mom2")
in_ap = (ana.radius < ana._radius_max_pix) & np.isfinite(np.asarray(ana.co_deproj)) & np.isfinite(disp)
DISP_EDGES = np.r_[0.0, np.nanpercentile(disp[in_ap], [100 / 3, 200 / 3]), np.inf]
print("dispersion edges [km/s]:", np.round(DISP_EDGES, 2))

disp_masks = {f"σ_HI {lo:.1f}–{hi:.1f}" if np.isfinite(hi) else f"σ_HI > {lo:.1f}":
              ana.selection_mask(dispersion_label="hi_mom2", dispersion_range_kms=(lo, hi))
              for lo, hi in zip(DISP_EDGES[:-1], DISP_EDGES[1:])}
show_masks(disp_masks)

dsp = compare({"all": {}, **{k: {"sel": m} for k, m in disp_masks.items()}},
              "HI velocity dispersion tertiles",
              colors=["k", "#9ecae1", "#4292c6", "#08306b"])

## C. Displaced CO (rotate / mirror)

These are the same transforms as the divergence chance-alignment null test, now applied to the shear
results. The shear field stays in the real frame. The displaced CO keeps its radial
distribution but loses its local correspondence with the velocity field.

Caveat: |shear| and CO both vary with radius, so a displaced map can keep part of
the trend through that shared radial structure alone. The second cell repeats
the comparison inside galactocentric annuli, where that is largely removed.

In [ ]:
MODES = {"rot180": "C3", "mirror_major": "C2", "mirror_minor": "C4"}
mir = compare({"real CO": {}, **{m: {"co_mode": m} for m in MODES}},
              "Displaced CO", colors=["C0", *MODES.values()])

In [ ]:
# within radial annuli (Result 4 only): real vs rot180
ANNULI = [(0, 900), (900, 1500), (1500, 2500), (2500, 4000)]
fig, axes = plt.subplots(1, len(ANNULI), figsize=(3.6 * len(ANNULI), 3.4), sharey=True)
for ax, (lo, hi) in zip(axes, ANNULI):
    pair = {m or "real CO": ana.shear_sensitivity(decim=DECIM, n_boot=N_BOOT,
                                                  radius_window_pc=(lo, hi), co_mode=m)
            for m in (None, "rot180")}
    plotting.plot_shear_sensitivity(pair, ax=ax, colors=["C0", "C3"])
    ax.set_title(f"{lo}-{hi} pc")
    print(f"{lo:5d}-{hi:<5d} pc  " + "  ".join(
        f"{k}: {v.rel_drop:5.2f} [{v.rel_drop_lo:.2f}, {v.rel_drop_hi:.2f}]" for k, v in pair.items()))
fig.tight_layout()
plt.show()

## Result 3 (divergence): CO fraction vs divergence across smoothing scales

This is Result 3 from the `reproduce_results_check` notebooks, CO-bright fraction vs divergence
at each smoothing FWHM. The first figure is the baseline, the same as those notebooks at `decim=2`. The
next three overlay the selections from checks A–C. These are the divergence curves of the
`resolution_study` calls above, so nothing is recomputed. Converging flows are negative divergence.

In [ ]:
# baseline (no selection), as in reproduce_results_check.ipynb
plotting.plot_resolution_study(lap["r3"]["all"], which="divergence")
plt.show()

# the selections from checks A-C
n = len(ana.config.analysis.smoothing_fwhm_pc)
for res in (lap, dsp, mir):
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2), sharey=True)
    for (lab, st), c in zip(res["r3"].items(), res["colors"]):
        plotting.plot_resolution_study(st, which="divergence", axes=axes, label=lab, color=c)
    for ax in axes:
        ax.axvline(0, color="0.6", lw=0.8, ls=":")
    fig.suptitle(f"Result 3 (divergence): {res['title']}", y=1.02)
    plt.show()

### Diagnostic: are the large-FWHM divergence spikes edge effects?

At 58 and 86 pc the divergence curves develop sharp peaks near |div| ≈ 80–100 km/s/kpc.
Several masks can produce such peaks:

* **NaN growth.** `gaussian_filter` turns every pixel within ~4σ of a NaN (strict-mask holes and the
  footprint edge) into NaN. At large FWHM the surviving pixels crowd against those holes, which is also
  where deprojection's zero-fill leaves spurious gradients.
* **CO coverage.** Result 3 counts pixels *without* CO coverage as CO-dark, so a bin dominated by
  pixels outside the CO footprint is pulled down, and one dominated by pixels inside it is pushed up.
* **Small numbers.** The extreme-divergence bins at large FWHM hold few pixels.

The first cell recomputes each smoothed divergence field (same calls as `resolution_study`) and
compares three curves: the Result 3 curve (all pixels in the aperture), CO-covered pixels only, and
CO-covered pixels in the **interior** (farther than `EDGE_FACTOR` × FWHM from any NaN in the smoothed field).
Pixel counts per bin are shown underneath. **If the spikes vanish in the interior curve, they are edge
effects. If their bins hold only a handful of pixels, they are noise.** The second cell maps where the
spike-range pixels sit. Set `DIAG_SEL` to one of the masks above (e.g. `disp_masks["σ_HI > 6.5"]`)
to diagnose a single selection's curve.

In [ ]:
from scipy import ndimage as nd
from exgal_flows import flows as _flows, statistics as _stats

an = ana.config.analysis
DIAG_SEL = None            # e.g. disp_masks["σ_HI > 6.5"] to diagnose one selection's curve
EDGE_FACTOR = 2.0          # edge band = within EDGE_FACTOR x FWHM of a NaN in the smoothed field
EDGE_MIN_PC = 20.0         # edge band width used for the unsmoothed field
SPIKE_DIV = (60.0, 130.0)  # |divergence| range (km/s/kpc) holding the large-FWHM spikes

sl = (slice(None, None, DECIM), slice(None, None, DECIM))
px_d = ana.pixel_scale_pc * DECIM
co_d = np.asarray(ana.co_deproj)[sl]
covered = np.isfinite(co_d)
bright = co_d > an.co_threshold_K
base = ana.radius[sl] < ana._radius_max_pix          # the Result 3 aperture
if DIAG_SEL is not None:
    base = base & np.asarray(DIAG_SEL, dtype=bool)[sl]
dv_edges = np.arange(*an.div_range, an.grad_bin_width)
hist_edges = np.r_[dv_edges, dv_edges[-1] + an.grad_bin_width]

# smoothed divergence per FWHM (decimated), its valid area, and the interior away from NaNs
diag = {}
for fwhm in an.smoothing_fwhm_pc:
    vsm = _flows.smoothed_velocity_field(ana.mom0_deproj, ana.velocity_deproj, ana._sigma_pix(fwhm))
    div = _flows.velocity_gradients(vsm, ana.pixel_scale_pc, ana.config.galaxy).divergence.value[sl]
    del vsm
    valid = np.isfinite(div)
    edge_pc = max(EDGE_FACTOR * fwhm, EDGE_MIN_PC)
    dist_pc = nd.distance_transform_edt(valid) * px_d     # distance to the nearest NaN
    diag[fwhm] = dict(div=div.astype(np.float32), valid=valid,
                      interior=valid & (dist_pc > edge_pc), edge_pc=edge_pc)
    del div, dist_pc


def _spike(d):
    a = np.abs(d["div"]) * 1e3
    return d["valid"] & base & (a >= SPIKE_DIV[0]) & (a < SPIKE_DIV[1])


SETS = {"Result 3 (all in aperture)": lambda d: base,
        "CO-covered only": lambda d: base & covered,
        "CO-covered, interior": lambda d: base & covered & d["interior"]}
SET_COLORS = ["k", "C0", "C2"]

n = len(diag)
fig, axes = plt.subplots(2, n, figsize=(3.4 * n, 6.4), sharex="col", sharey="row")
for k, (fwhm, d) in enumerate(diag.items()):
    g = d["div"].ravel()
    for (lab, fn), c in zip(SETS.items(), SET_COLORS):
        m = fn(d).ravel()
        cur = _stats.co_fraction_vs_gradient(g, co_d.ravel(), an.co_threshold_K, dv_edges, m)
        axes[0, k].plot(cur.centers * 1e3, cur.fraction, color=c, label=lab)
        cnt, _ = np.histogram(g[m & np.isfinite(g)], hist_edges)
        axes[1, k].step(cur.centers * 1e3, np.where(cnt > 0, cnt, np.nan), where="mid", color=c)
    for ax in axes[:, k]:
        for s in (-1, 1):
            ax.axvspan(*sorted(s * np.array(SPIKE_DIV)), color="0.9", zorder=0)
        ax.axvline(0, color="0.6", lw=0.8, ls=":")
    axes[0, k].set_title("no smoothing" if fwhm == 0 else f"FWHM = {fwhm:.0f} pc")
    axes[1, k].set_yscale("log")
    axes[1, k].set_xlabel("Divergence [km/s/kpc]")
axes[0, 0].set_ylabel("Fraction of pixels with CO")
axes[1, 0].set_ylabel(f"pixels per bin (decim={DECIM})")
axes[0, 0].legend(frameon=False, fontsize=7)
fig.suptitle(f"Smoothed-divergence edge diagnostic (grey = spike range |div| {SPIKE_DIV[0]:g}-{SPIKE_DIV[1]:g}; "
             f"interior = > {EDGE_FACTOR:g} x FWHM from NaN)", y=1.0)
fig.tight_layout()
plt.show()


def _pct(x, of):
    return 100.0 * np.count_nonzero(x) / max(np.count_nonzero(of), 1)


def _frac(m):
    return np.count_nonzero(m & bright) / max(np.count_nonzero(m), 1)


print("Spike-range pixels vs all valid pixels in the aperture")
print(f"{'FWHM':>6s} {'edge':>6s} {'valid':>8s} {'spike':>7s} | {'% in edge band':>15s} | "
      f"{'% no CO cover':>14s} | CO frac in spike range: R3 / covered / interior")
print(f"{'[pc]':>6s} {'[pc]':>6s} {'[px]':>8s} {'[px]':>7s} | {'valid   spike':>15s} | {'valid   spike':>14s} |")
for fwhm, d in diag.items():
    v = d["valid"] & base
    sp = _spike(d)
    edge = ~d["interior"]
    print(f"{fwhm:6.0f} {d['edge_pc']:6.0f} {np.count_nonzero(v):8d} {np.count_nonzero(sp):7d} | "
          f"{_pct(v & edge, v):6.1f}  {_pct(sp & edge, sp):6.1f} | "
          f"{_pct(v & ~covered, v):6.1f}  {_pct(sp & ~covered, sp):6.1f} | "
          f"{_frac(sp):.3f} / {_frac(sp & covered):.3f} / {_frac(sp & covered & d['interior']):.3f}")

In [ ]:
# where the spike-range pixels sit, for the smoothed fields
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

MAP_FWHMS = [fw for fw in diag if fw >= 25]
DIV_VLIM = 100.0           # km/s/kpc colour limit for the divergence maps

# BBOX (from the helper cell) on the decimated grid, and its extent in kpc
bb = tuple(slice(s.start // DECIM, -(-s.stop // DECIM)) for s in BBOX)
ext = [(bb[1].start * DECIM - _cx) * _kpc, (bb[1].stop * DECIM - _cx) * _kpc,
       (bb[0].start * DECIM - _cy) * _kpc, (bb[0].stop * DECIM - _cy) * _kpc]
CAT_COLORS = ["0.35", "0.75", "#fdbb84", "0.97"]
CAT_LABELS = ["NaN in smoothed field / outside aperture", "valid, no CO coverage",
              "CO-covered, edge band", "CO-covered, interior"]

for fwhm in MAP_FWHMS:
    d = diag[fwhm]
    cat = np.zeros(d["div"].shape, np.int8)
    cat[d["valid"]] = 1
    cat[d["valid"] & covered] = 2
    cat[d["interior"] & covered] = 3
    cat[~base] = 0

    fig, (axl, axr) = plt.subplots(1, 2, figsize=(13.5, 6.2))
    im = axl.imshow(np.where(base, d["div"] * 1e3, np.nan)[bb], origin="lower", extent=ext,
                    cmap="RdBu_r", vmin=-DIV_VLIM, vmax=DIV_VLIM, interpolation="nearest")
    axl.set_facecolor("0.8")
    axl.contour(CO_BRIGHT, levels=[0.25], extent=EXTENT, origin="lower", colors="k", linewidths=0.6)
    fig.colorbar(im, ax=axl, shrink=0.8, label="smoothed divergence [km/s/kpc]")
    axl.set_title(f"FWHM = {fwhm:.0f} pc: divergence (black: CO-bright fraction 0.25)")

    axr.imshow(cat[bb], origin="lower", extent=ext, cmap=ListedColormap(CAT_COLORS),
               vmin=-0.5, vmax=3.5, interpolation="nearest")
    ii, jj = np.nonzero(_spike(d))
    x, y, b = (jj * DECIM - _cx) * _kpc, (ii * DECIM - _cy) * _kpc, bright[ii, jj]
    axr.scatter(x[~b], y[~b], s=3, c="C0", lw=0, rasterized=True)
    axr.scatter(x[b], y[b], s=5, c="C3", lw=0, rasterized=True)
    handles = [Patch(color=c, label=l) for c, l in zip(CAT_COLORS, CAT_LABELS)] + [
        Patch(color="C0", label=f"|div| {SPIKE_DIV[0]:g}-{SPIKE_DIV[1]:g}, CO-dark"),
        Patch(color="C3", label=f"|div| {SPIKE_DIV[0]:g}-{SPIKE_DIV[1]:g}, CO-bright")]
    axr.legend(handles=handles, fontsize=7, loc="upper left", framealpha=0.9)
    axr.set_title(f"spike-range pixels; edge band = {d['edge_pc']:.0f} pc from NaN")
    for ax in (axl, axr):
        ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:])
        ax.set_xlabel("x [kpc]"); ax.set_ylabel("y [kpc]")
    fig.tight_layout()
    plt.show()